# Generación de Datos Sintéticos
### Métodos, Validación e Implementación

**Magíster en Ciencia de Datos e Inteligencia Artificial**
Universidad Andrés Bello

*Maidana, J.P. (2026). Apunte — versión Jupyter Notebook.*

---

Versión **ejecutable** del apunte. Cada método (paramétrico, perturbación, SMOTE) se implementa y se aplica sobre datos concretos, con visualizaciones que comparan los datos reales con los sintéticos.

> 💡 Busca los comentarios **`👉 EXPERIMENTA`**: marcan parámetros que vale la pena cambiar (magnitud del ruido, número de muestras, etc.).

**Sobre las dependencias:** el cuaderno usa `numpy`, `pandas`, `matplotlib`, `scipy` y `scikit-learn`. La sección de SMOTE implementa el algoritmo **desde cero** (para que se entienda) y además muestra cómo usar `imbalanced-learn`, que viene preinstalado en Google Colab.

## Configuración del entorno

Importamos librerías, fijamos estilo y definimos utilidades de dibujo. **Ejecuta esta celda primero.**

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

np.random.seed(42)

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

AZUL    = "#2c7fb8"   # datos reales
NARANJA = "#fe9929"   # datos sintéticos
VERDE   = "#41ab5d"
ROJO    = "#e34a33"

def caja(ax, x, y, w, h, texto, color, fontsize=10):
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.12",
                         linewidth=1.6, edgecolor="#37474f", facecolor=color, alpha=0.92)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, texto, ha="center", va="center", fontsize=fontsize, zorder=5)

print("Entorno listo ✔")

---
# 1. Introducción: ¿por qué generar datos sintéticos?

En ciencia de datos enfrentamos a menudo un dilema: tenemos un dataset limitado pero necesitamos más datos para entrenar modelos, evaluar algoritmos o probar hipótesis. La generación de datos sintéticos ofrece una solución.

> ### 📦 Definición — Datos Sintéticos
> Son datos **artificialmente generados** que imitan las propiedades estadísticas de datos reales, pero que no provienen directamente de observaciones del mundo real.
>
> **Objetivo:** crear nuevos datos que sean *realistas* (parecidos a los reales) pero *artificiales* (generados computacionalmente).

## 1.1 Motivaciones

In [ ]:
# Diagrama de motivaciones
fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(-7, 7); ax.set_ylim(-4, 4); ax.axis("off")

# Centro
centro = plt.Circle((0, 0), 1.3, facecolor="#0d2741", edgecolor="#37474f", lw=2)
ax.add_patch(centro)
ax.text(0, 0, "Datos\nSintéticos", ha="center", va="center", color="white",
        fontsize=12, weight="bold", zorder=5)

motivaciones = [
    (4, 2.5, "Aumentar\ndatos",        "#bfe3bf"),
    (4, 0,   "Privacidad\ny anonimato","#ffd9a8"),
    (4, -2.5,"Evaluar\nmodelos",       "#fff2b3"),
    (-4, 2.5,"Balancear\nclases",      "#e3c8f0"),
    (-4, 0,  "Compartir\ndatasets",    "#f3b0b0"),
    (-4, -2.5,"Simular\nescenarios",   "#bfe6ef"),
]
for x, y, t, c in motivaciones:
    caja(ax, x - 1.3, y - 0.6, 2.6, 1.2, t, c, fontsize=9.5)
    ax.annotate("", xy=(x - 1.3 if x > 0 else x + 1.3, y),
                xytext=(np.sign(x)*1.3, np.sign(y)*0.6 if y != 0 else 0),
                arrowprops=dict(arrowstyle="-|>", lw=1.8, color="#0d2741"))

ax.set_title("Seis motivaciones para generar datos sintéticos", fontsize=13, weight="bold")
plt.tight_layout(); plt.show()

**1. Aumentar datos de entrenamiento (*data augmentation*).** Más datos suelen mejorar el desempeño. Si tienes 1,000 observaciones pero necesitas 10,000, generas 9,000 sintéticas.
*Ejemplo:* clasificar rayos X con sólo 500 casos raros de una enfermedad → generas 5,000 casos sintéticos para balancear.

**2. Proteger privacidad y anonimato.** Los datos reales pueden contener información sensible. Los sintéticos mantienen propiedades estadísticas sin exponer individuos reales.
*Ejemplo:* un hospital comparte datos para investigación sin revelar pacientes reales.

**3. Evaluar y validar modelos.** Cuando conoces la "verdad" (porque tú la generaste), puedes evaluar si un modelo recupera parámetros o relaciones conocidas.
*Ejemplo:* generas datos con correlación exacta $\rho = 0.7$ y verificas si el modelo la detecta.

**4. Balancear clases desbalanceadas.** Generas observaciones sintéticas de la clase minoritaria.
*Ejemplo:* fraude (99% legítimas, 1% fraude) → generas más casos de fraude para entrenar mejor.

**5. Compartir datasets públicamente.** Publicas una versión sintética cuando los datos originales son confidenciales.

**6. Simular escenarios hipotéticos.** Explorar "¿qué pasaría si...?" variando condiciones.

> ### ❗ Principio fundamental
> Los datos sintéticos deben ser **indistinguibles** de los reales en las propiedades estadísticas relevantes, pero **no** deben ser copias exactas (eso no protege privacidad ni genera información nueva).
>
> **Balance:** suficientemente realistas para ser útiles, suficientemente diferentes para ser seguros.

## 1.2 Diferencia con bootstrap

| | **Bootstrap** | **Datos Sintéticos** |
|---|---|---|
| **Origen** | Remuestreo con reemplazo de datos reales | Generación desde un modelo estadístico |
| **Contenido** | Sólo observaciones que ya existen | Nuevas observaciones nunca vistas |
| **Objetivo** | Estimar variabilidad de estadísticos | Aumentar dataset o proteger privacidad |
| **Realismo** | 100% real (son datos observados) | Depende de qué tan bien el modelo capture la realidad |

> ### 📝 Lo que aprenderás
> 1. Por qué y cuándo generar datos sintéticos.
> 2. Método paramétrico: ajustar distribución y muestrear.
> 3. Método de perturbación: añadir ruido a datos reales.
> 4. Métodos avanzados: SMOTE y GANs (introducción).
> 5. Cómo validar que los datos sintéticos son buenos.
> 6. Métricas de similitud y utilidad, e implementación en Python.

---
# 2. Método 1: Generación Paramétrica

## 2.1 Concepto

El enfoque más directo: ajustar una distribución probabilística a los datos reales y muestrear de ella.

> ### 📦 Método paramétrico
> **Pasos:** (1) analizar los datos reales, (2) ajustar una distribución paramétrica, (3) estimar sus parámetros, (4) generar nuevas muestras.
>
> *Ejemplo univariado:* datos = edades de clientes → ajustar $X \sim N(\mu, \sigma^2)$ → estimar $\hat{\mu}=45,\ \hat{\sigma}=12$ → muestrear de $N(45, 12^2)$.

## 2.2 Univariado: una variable

Ajustamos varias distribuciones candidatas a los datos reales y elegimos la mejor por **AIC** (menor es mejor).

In [ ]:
from scipy import stats

# Datos reales (simulados para el ejemplo): ingresos con sesgo a la derecha
np.random.seed(42)
datos_reales = np.random.lognormal(mean=10.5, sigma=0.5, size=500)

print("Paso 1 — Análisis de datos reales")
print(f"  n = {len(datos_reales)} | Media = {datos_reales.mean():.0f} | "
      f"Mediana = {np.median(datos_reales):.0f} | Desv = {datos_reales.std():.0f}")

# Paso 2 — Ajustar distribuciones candidatas y comparar por AIC
distribuciones = {"Normal": stats.norm, "Lognormal": stats.lognorm,
                  "Exponencial": stats.expon, "Gamma": stats.gamma}
mejores_params, aic_scores = {}, {}
print("\nPaso 2 — Ajuste (AIC = -2·loglik + 2k):")
for nombre, dist in distribuciones.items():
    params = dist.fit(datos_reales)
    mejores_params[nombre] = params
    log_lik = np.sum(dist.logpdf(datos_reales, *params))
    aic_scores[nombre] = -2 * log_lik + 2 * len(params)
    print(f"  {nombre:12s} AIC = {aic_scores[nombre]:.1f}")

mejor_dist = min(aic_scores, key=aic_scores.get)     # (el apunte tenía aquí un typo: 'mejor-dist')
print(f"\n  -> Mejor distribución: {mejor_dist}")

# Paso 3 — Generar datos sintéticos desde la distribución elegida
dist_elegida = distribuciones[mejor_dist]
params_elegidos = mejores_params[mejor_dist]
datos_sinteticos = dist_elegida.rvs(*params_elegidos, size=1000, random_state=123)

print("\nPaso 3 — Datos sintéticos")
print(f"  Media = {datos_sinteticos.mean():.0f} | Mediana = {np.median(datos_sinteticos):.0f} | "
      f"Desv = {datos_sinteticos.std():.0f}")

In [ ]:
# Visualización: ajuste y comparación real vs sintético
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
xx = np.linspace(datos_reales.min(), datos_reales.max(), 300)

axes[0].hist(datos_reales, bins=40, density=True, alpha=0.6, color=AZUL, label="Datos reales")
axes[0].plot(xx, dist_elegida.pdf(xx, *params_elegidos), color=ROJO, lw=2.5,
             label=f"{mejor_dist} ajustada")
axes[0].set_title("Datos reales + distribución ajustada"); axes[0].legend()
axes[0].set_xlabel("Ingreso")

axes[1].hist(datos_reales, bins=40, density=True, alpha=0.55, color=AZUL, label="Real")
axes[1].hist(datos_sinteticos, bins=40, density=True, alpha=0.55, color=NARANJA, label="Sintético")
axes[1].set_title("Real vs Sintético"); axes[1].legend(); axes[1].set_xlabel("Ingreso")
plt.tight_layout(); plt.show()

## 2.3 Multivariado: múltiples variables

Con varias variables debemos capturar no sólo las distribuciones marginales sino también las **correlaciones**.

> ### 🔧 Generación multivariada normal
> Si las variables son aproximadamente normales multivariadas: (1) calcular el vector de medias $\boldsymbol{\mu}$, (2) la matriz de covarianza $\boldsymbol{\Sigma}$, (3) generar $\mathbf{X} \sim N_p(\hat{\boldsymbol{\mu}}, \hat{\boldsymbol{\Sigma}})$.

In [ ]:
def generar_sinteticos_multivariados(datos_reales, n_sinteticos, random_state=None):
    '''Genera datos sintéticos multivariados usando una normal multivariada.'''
    mu = datos_reales.mean(axis=0)
    Sigma = np.cov(datos_reales, rowvar=False)
    rng = np.random.default_rng(random_state)
    datos_sinteticos = rng.multivariate_normal(mu, Sigma, size=n_sinteticos)
    return datos_sinteticos, mu, Sigma

# Dataset multivariado de ejemplo (3 variables correlacionadas)
np.random.seed(42)
mu_real = np.array([50, 100, 75])
Sigma_real = np.array([[100, 80, 30],
                       [ 80, 200, 50],
                       [ 30,  50, 150]])
datos_reales_multi = np.random.multivariate_normal(mu_real, Sigma_real, size=500)

datos_sint_multi, mu_est, Sigma_est = generar_sinteticos_multivariados(
    datos_reales_multi, n_sinteticos=1000, random_state=123)

print("Vector de medias:")
print(f"  Real:      {np.round(mu_real, 1)}")
print(f"  Sintético: {np.round(datos_sint_multi.mean(axis=0), 1)}")

corr_real = np.corrcoef(datos_reales_multi, rowvar=False)
corr_sint = np.corrcoef(datos_sint_multi, rowvar=False)
print("\nCorrelación real (triángulo superior):", np.round(corr_real[np.triu_indices(3, 1)], 3))
print("Correlación sintética:                 ", np.round(corr_sint[np.triu_indices(3, 1)], 3))

In [ ]:
# Visualización: matrices de correlación real vs sintética
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, C, t in [(axes[0], corr_real, "Real"), (axes[1], corr_sint, "Sintético")]:
    ax.imshow(C, vmin=-1, vmax=1, cmap="coolwarm")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{C[i, j]:.2f}", ha="center", va="center", fontsize=11)
    ax.set_title(f"Correlación — {t}")
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(["V1", "V2", "V3"]); ax.set_yticklabels(["V1", "V2", "V3"])
    ax.grid(False)
plt.tight_layout(); plt.show()
print("Las correlaciones sintéticas reproducen de cerca las reales.")

## 2.4 Variables mixtas (continuas y categóricas)

> ### 🔧 Estrategias para variables mixtas
> - **Estratificación:** dividir por combinación de categorías y generar continuas dentro de cada estrato.
> - **Cópulas:** transformar a uniformes $[0,1]$, modelar dependencias con una cópula y transformar de vuelta.
> - **Condicional:** generar las categorías con sus proporciones observadas y luego las continuas condicionadas a la categoría.

Implementamos el enfoque **condicional**:

In [ ]:
def generar_sinteticos_mixtos(datos_reales, columnas_categoricas, n_sinteticos, random_state=None):
    '''Genera datos sintéticos con variables continuas y categóricas (enfoque condicional):
       primero las categóricas con sus proporciones, luego las continuas condicionadas.'''
    rng = np.random.default_rng(random_state)
    datos_sint = {}

    # Paso 1: variables categóricas según proporciones observadas
    for col_cat in columnas_categoricas:
        valores, frecuencias = np.unique(datos_reales[col_cat], return_counts=True)
        probabilidades = frecuencias / len(datos_reales)
        datos_sint[col_cat] = rng.choice(valores, size=n_sinteticos, p=probabilidades)

    # Paso 2: continuas condicionadas a la primera categórica
    columnas_continuas = [c for c in datos_reales.columns if c not in columnas_categoricas]
    if columnas_categoricas:
        primera_cat = columnas_categoricas[0]
        for valor_cat in np.unique(datos_reales[primera_cat]):
            subset = datos_reales.loc[datos_reales[primera_cat] == valor_cat, columnas_continuas]
            if len(subset) == 0:
                continue
            mu, Sigma = subset.mean().values, subset.cov().values
            mask_sint = datos_sint[primera_cat] == valor_cat
            n_grupo = mask_sint.sum()
            if n_grupo > 0:
                sint_cont = rng.multivariate_normal(mu, Sigma, size=n_grupo)
                for j, col in enumerate(columnas_continuas):
                    if col not in datos_sint:
                        datos_sint[col] = np.zeros(n_sinteticos)
                    datos_sint[col][mask_sint] = sint_cont[:, j]
    else:
        mu = datos_reales[columnas_continuas].mean().values
        Sigma = datos_reales[columnas_continuas].cov().values
        sint_cont = rng.multivariate_normal(mu, Sigma, size=n_sinteticos)
        for j, col in enumerate(columnas_continuas):
            datos_sint[col] = sint_cont[:, j]

    return pd.DataFrame(datos_sint)

# Dataset mixto de ejemplo
np.random.seed(0)
df_mixto = pd.DataFrame({
    "edad":     np.random.normal(40, 10, 400),
    "ingreso":  np.random.normal(50000, 15000, 400),
    "segmento": np.random.choice(["A", "B", "C"], 400, p=[0.5, 0.3, 0.2]),
})

sint_mixto = generar_sinteticos_mixtos(df_mixto, ["segmento"], n_sinteticos=600, random_state=1)

print("Proporciones de 'segmento':")
print("  Real:     ", df_mixto["segmento"].value_counts(normalize=True).round(3).to_dict())
print("  Sintético:", sint_mixto["segmento"].value_counts(normalize=True).round(3).to_dict())
print(f"\nMedia de 'ingreso':  real = {df_mixto.ingreso.mean():.0f}  |  "
      f"sintético = {sint_mixto.ingreso.mean():.0f}")
sint_mixto.head()

---
# 3. Método 2: Perturbación (*noise addition*)

## 3.1 Concepto

En lugar de ajustar una distribución, tomamos datos reales y les añadimos **ruido controlado**.

> ### 📦 Método de perturbación
> **Idea:** preservar la estructura de los datos reales pero añadir variabilidad para proteger identidades y crear versiones ligeramente diferentes.
>
> **Fórmula básica:** $X_{sint} = X_{real} + \epsilon$, con $\epsilon \sim N(0, \sigma^2)$.
>
> - $\sigma$ pequeño → sintéticos muy parecidos a los reales (menos privacidad).
> - $\sigma$ grande → sintéticos muy diferentes (pueden perder utilidad).

## 3.2 Perturbación con escala adaptativa

Para que el ruido sea apropiado para cada variable, lo escalamos según su variabilidad:

$$ X_{j,sint} = X_{j,real} + \epsilon_j, \qquad \epsilon_j \sim N\!\left(0,\ (\alpha \cdot s_j)^2\right) $$

donde $s_j$ es la desviación estándar de la variable $j$ y $\alpha$ es el factor de perturbación (típicamente entre 0.05 y 0.3): se añade ruido del $\alpha\times100\%$ de la variabilidad original.

In [ ]:
def generar_por_perturbacion(datos_reales, factor_ruido=0.1, n_copias=1, random_state=None):
    '''Genera datos sintéticos mediante perturbación: X_sint = X_real + ruido escalado.'''
    rng = np.random.default_rng(random_state)

    es_dataframe = isinstance(datos_reales, pd.DataFrame)
    if es_dataframe:
        columnas = datos_reales.columns
        datos_array = datos_reales.values
    else:
        datos_array = np.asarray(datos_reales)

    n, p = datos_array.shape
    stds = np.std(datos_array, axis=0)

    datos_sint_list = []
    for _ in range(n_copias):
        ruido = rng.normal(0, 1, size=(n, p)) * stds * factor_ruido
        datos_sint_list.append(datos_array + ruido)
    datos_sinteticos = np.vstack(datos_sint_list)

    if es_dataframe:
        datos_sinteticos = pd.DataFrame(datos_sinteticos, columns=columnas)
    return datos_sinteticos

# Datos reales 2D correlacionados (para visualizar el efecto)
real_2d = np.random.default_rng(0).multivariate_normal([5, 8], [[1, 0.8], [0.8, 1]], size=200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, alpha in zip(axes, [0.05, 0.15, 0.30]):   # 👉 EXPERIMENTA: cambia estos factores
    sint = generar_por_perturbacion(real_2d, factor_ruido=alpha, random_state=1)
    ax.scatter(real_2d[:, 0], real_2d[:, 1], alpha=0.5, color=AZUL, s=25, label="Real")
    ax.scatter(sint[:, 0], sint[:, 1], alpha=0.5, color=NARANJA, s=25, label="Sintético")
    ax.set_title(f"α = {alpha}  (ruido = {int(alpha*100)}%)")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print("A mayor α, los sintéticos se alejan más de los reales (más privacidad, menos fidelidad).")

## 3.3 Perturbación con preservación de restricciones

A veces las variables tienen restricciones que debemos respetar: **no negatividad** (edad, ingreso), **rangos** (porcentajes $[0,100]$), o **enteros** (número de hijos).

In [ ]:
def perturbacion_con_restricciones(datos_reales, factor_ruido=0.1, restricciones=None, random_state=None):
    '''Perturbación que respeta restricciones por columna:
       'positive' | ('bound', min, max) | 'integer'.'''
    rng = np.random.default_rng(random_state)
    datos_array = np.asarray(datos_reales, dtype=float)
    n, p = datos_array.shape
    stds = np.std(datos_array, axis=0)

    datos_sint = datos_array + rng.normal(0, 1, size=(n, p)) * stds * factor_ruido

    if restricciones:
        for col_idx, restriccion in restricciones.items():
            if restriccion == "positive":
                datos_sint[:, col_idx] = np.abs(datos_sint[:, col_idx])
            elif isinstance(restriccion, tuple) and restriccion[0] == "bound":
                _, vmin, vmax = restriccion
                datos_sint[:, col_idx] = np.clip(datos_sint[:, col_idx], vmin, vmax)
            elif restriccion == "integer":
                datos_sint[:, col_idx] = np.round(datos_sint[:, col_idx])
    return datos_sint

# Datos con restricciones naturales: edad(+), ingreso(+), satisfacción[0-100], n_hijos(entero)
rng = np.random.default_rng(0)
real_restr = np.column_stack([
    rng.normal(40, 12, 300),       # edad
    rng.normal(50000, 15000, 300), # ingreso
    rng.uniform(0, 100, 300),      # satisfacción
    rng.integers(0, 5, 300),       # n_hijos
])
restricciones = {0: "positive", 1: "positive", 2: ("bound", 0, 100), 3: "integer"}
sint_restr = perturbacion_con_restricciones(real_restr, factor_ruido=0.2,
                                            restricciones=restricciones, random_state=1)

cols = ["edad", "ingreso", "satisfacción", "n_hijos"]
print(f"{'variable':14s}{'min':>10s}{'max':>10s}   ¿restricción respetada?")
print(f"  edad        {sint_restr[:,0].min():10.2f}{sint_restr[:,0].max():10.2f}   {'sí, ≥ 0' if sint_restr[:,0].min()>=0 else 'NO'}")
print(f"  ingreso     {sint_restr[:,1].min():10.0f}{sint_restr[:,1].max():10.0f}   {'sí, ≥ 0' if sint_restr[:,1].min()>=0 else 'NO'}")
print(f"  satisfacción{sint_restr[:,2].min():10.2f}{sint_restr[:,2].max():10.2f}   {'sí, en [0,100]' if sint_restr[:,2].min()>=0 and sint_restr[:,2].max()<=100 else 'NO'}")
es_entero = np.allclose(sint_restr[:,3], np.round(sint_restr[:,3]))
print(f"  n_hijos     {sint_restr[:,3].min():10.0f}{sint_restr[:,3].max():10.0f}   {'sí, entero' if es_entero else 'NO'}")

---
# 4. Métodos Avanzados (Introducción)

## 4.1 SMOTE para datos desbalanceados

> ### 📦 SMOTE (*Synthetic Minority Over-sampling Technique*)
> Diseñado para balancear clases minoritarias en clasificación.
>
> **Algoritmo:** para cada observación de la clase minoritaria, (1) encontrar sus $k$ vecinos más cercanos (típico $k=5$), (2) elegir uno al azar, (3) generar un punto sintético en la línea que los conecta:
> $$ x_{nuevo} = x + \lambda \cdot (x_{vecino} - x), \qquad \lambda \sim \text{Uniforme}(0, 1) $$
>
> **Resultado:** las nuevas observaciones quedan "entre" observaciones reales de la clase minoritaria.

Lo implementamos desde cero para ver el algoritmo en acción:

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.datasets import make_classification
from collections import Counter

def smote_manual(X_min, n_generar, k=5, random_state=None):
    '''SMOTE desde cero: interpola entre cada punto minoritario y uno de sus k vecinos.'''
    rng = np.random.default_rng(random_state)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X_min)   # +1 porque el vecino 0 es el punto mismo
    _, idx = nn.kneighbors(X_min)
    nuevos = np.empty((n_generar, X_min.shape[1]))
    for s in range(n_generar):
        i = rng.integers(len(X_min))
        vecino = idx[i, rng.integers(1, k + 1)]           # vecino al azar (excluye col 0)
        lam = rng.random()
        nuevos[s] = X_min[i] + lam * (X_min[vecino] - X_min[i])
    return nuevos

def cuenta(a):
    v, c = np.unique(a, return_counts=True)
    return dict(zip(v.tolist(), c.tolist()))

# Dataset desbalanceado (90% / 10%)
X, y = make_classification(n_samples=1000, n_features=2, n_informative=2, n_redundant=0,
                           n_clusters_per_class=1, weights=[0.9, 0.1], random_state=42)
print("Distribución original:", cuenta(y))

X_min = X[y == 1]
n_generar = (y == 0).sum() - (y == 1).sum()
X_nuevos = smote_manual(X_min, n_generar, k=5, random_state=42)
X_bal = np.vstack([X, X_nuevos])
y_bal = np.concatenate([y, np.ones(len(X_nuevos), dtype=int)])
print("Distribución balanceada:", cuenta(y_bal))

In [ ]:
# Visualización: antes vs después de SMOTE
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(X[y == 0, 0], X[y == 0, 1], alpha=0.4, color=AZUL, s=20, label="Mayoritaria")
axes[0].scatter(X[y == 1, 0], X[y == 1, 1], alpha=0.8, color=ROJO, s=30, label="Minoritaria")
axes[0].set_title(f"Antes  {cuenta(y)}"); axes[0].legend()

axes[1].scatter(X[y == 0, 0], X[y == 0, 1], alpha=0.4, color=AZUL, s=20, label="Mayoritaria")
axes[1].scatter(X_nuevos[:, 0], X_nuevos[:, 1], alpha=0.5, color=VERDE, s=20, marker="x",
                label="Minoritaria sintética")
axes[1].scatter(X[y == 1, 0], X[y == 1, 1], alpha=0.9, color=ROJO, s=30, label="Minoritaria real")
axes[1].set_title(f"Después de SMOTE  {cuenta(y_bal)}"); axes[1].legend()
plt.tight_layout(); plt.show()
print("Los puntos sintéticos (verdes) rellenan el espacio entre los minoritarios reales (rojos).")

In [ ]:
# En la práctica se usa imbalanced-learn (viene preinstalado en Google Colab)
try:
    from imblearn.over_sampling import SMOTE
    X_res, y_res = SMOTE(random_state=42).fit_resample(X, y)
    print("Con imbalanced-learn:")
    print("  Distribución tras SMOTE:", cuenta(y_res))
except ImportError:
    print("imbalanced-learn no está instalado en este entorno.")
    print("Instálalo con:  pip install imbalanced-learn   (en Colab ya viene incluido).")
    print("La implementación 'smote_manual' de arriba produce el mismo resultado.")

## 4.2 GANs: *Generative Adversarial Networks*

> ### 📝 GANs para datos tabulares
> Las GANs son el estado del arte en generación de datos sintéticos, especialmente para datos complejos.
>
> **Idea básica:** un **generador** aprende a crear datos sintéticos y un **discriminador** aprende a distinguir reales de sintéticos; entrenan en competencia (el generador trata de engañar al discriminador).
>
> **Ventajas:** capturan relaciones no lineales muy complejas, no requieren supuestos distribucionales, son estado del arte para imágenes y tablas.
> **Desventajas:** mucho más tiempo de entrenamiento, inestables, necesitan datasets grandes ($>$10,000 obs.) y son menos interpretables.
>
> **Librerías:** `SDV` (incluye CTGAN y TVAE), `CTGAN`, `synthcity`. Las GANs están más allá del alcance de este apunte, pero representan la frontera actual.

---
# 5. Validación de Datos Sintéticos

Una vez generados, debemos validar que son **realistas** y **útiles**.

## 5.1 Dimensiones de evaluación

| Dimensión | Pregunta |
|---|---|
| **Fidelidad** | ¿Qué tan parecidos son a los reales estadísticamente? |
| **Utilidad** | ¿Sirven para el propósito (entrenar un modelo, análisis)? |
| **Privacidad** | ¿Protegen identidades individuales? |
| **Diversidad** | ¿Cubren el espacio de los reales sin ser copias? |

## 5.2 Métricas de fidelidad

> ### 📦 Métricas estadísticas
> - **Distancia entre distribuciones:** Kolmogorov-Smirnov $D = \sup_x |F_{real}(x) - F_{sint}(x)|$; Wasserstein (*Earth Mover's Distance*).
> - **Comparación de momentos:** $\text{Error\_Media} = |\mu_{real} - \mu_{sint}| / |\mu_{real}|$.
> - **Correlaciones:** $\text{Error} = \|\mathbf{Corr}_{real} - \mathbf{Corr}_{sint}\|_F$ (norma de Frobenius).
> - **Pruebas estadísticas:** test KS ($H_0$: misma distribución) y test $\chi^2$ para categóricas.

In [ ]:
from scipy.stats import ks_2samp, wasserstein_distance

def validar_datos_sinteticos(datos_reales, datos_sinteticos, nombres_vars=None):
    '''Suite de validación: momentos, KS, Wasserstein y error de correlación, con una calificación final.'''
    if isinstance(datos_reales, pd.DataFrame):
        nombres_vars = list(datos_reales.columns)
        reales, sint = datos_reales.values, datos_sinteticos.values
    else:
        reales, sint = np.asarray(datos_reales), np.asarray(datos_sinteticos)
        if nombres_vars is None:
            nombres_vars = [f"Var{i+1}" for i in range(reales.shape[1])]

    resultados = {}
    print("=" * 64)
    print("VALIDACIÓN DE DATOS SINTÉTICOS")
    print("=" * 64)

    print("\n1) COMPARACIÓN DE MOMENTOS")
    for idx, nombre in enumerate(nombres_vars):
        mr, ms = reales[:, idx].mean(), sint[:, idx].mean()
        sr, ss = reales[:, idx].std(),  sint[:, idx].std()
        err_m = abs(mr - ms) / (abs(mr) + 1e-10)
        err_s = abs(sr - ss) / (sr + 1e-10)
        print(f"  {nombre:8s} media: {mr:8.2f} vs {ms:8.2f} (err {err_m*100:4.1f}%) | "
              f"std: {sr:7.2f} vs {ss:7.2f} (err {err_s*100:4.1f}%)")
        resultados[f"{nombre}_error_media"] = err_m

    print("\n2) TEST KOLMOGOROV-SMIRNOV (H0: misma distribución)")
    for idx, nombre in enumerate(nombres_vars):
        stat, pval = ks_2samp(reales[:, idx], sint[:, idx])
        print(f"  {nombre:8s} KS = {stat:.4f}, p = {pval:.4f}  "
              f"{'OK' if pval > 0.05 else '✗ difieren'}")
        resultados[f"{nombre}_ks_pval"] = pval

    print("\n3) DISTANCIA DE WASSERSTEIN")
    for idx, nombre in enumerate(nombres_vars):
        print(f"  {nombre:8s} {wasserstein_distance(reales[:, idx], sint[:, idx]):.4f}")

    if reales.shape[1] > 1:
        diff = np.abs(np.corrcoef(reales, rowvar=False) - np.corrcoef(sint, rowvar=False))
        print(f"\n4) CORRELACIONES — error Frobenius: {np.linalg.norm(diff, 'fro'):.4f} | "
              f"error promedio: {diff[np.triu_indices_from(diff, 1)].mean():.4f}")

    errores_media = [v for k, v in resultados.items() if "error_media" in k]
    pvals_ks = [v for k, v in resultados.items() if "ks_pval" in k]
    prop_pass = np.mean([p > 0.05 for p in pvals_ks])

    print("\n" + "=" * 64)
    if np.mean(errores_media) < 0.05 and prop_pass > 0.8:
        print("CALIFICACIÓN: EXCELENTE")
    elif np.mean(errores_media) < 0.10 and prop_pass > 0.6:
        print("CALIFICACIÓN: BUENA")
    elif np.mean(errores_media) < 0.20:
        print("CALIFICACIÓN: REGULAR — considerar mejorar la generación")
    else:
        print("CALIFICACIÓN: POBRE — datos sintéticos inadecuados")
    return resultados

# Validamos los sintéticos multivariados generados en la Sección 2
_ = validar_datos_sinteticos(datos_reales_multi, datos_sint_multi)

In [ ]:
# Apoyo visual a la validación: distribuciones marginales real vs sintético
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx in range(3):
    axes[idx].hist(datos_reales_multi[:, idx], bins=30, density=True, alpha=0.55,
                   color=AZUL, label="Real")
    axes[idx].hist(datos_sint_multi[:, idx], bins=30, density=True, alpha=0.55,
                   color=NARANJA, label="Sintético")
    axes[idx].set_title(f"Variable {idx+1}"); axes[idx].legend()
plt.tight_layout(); plt.show()

## 5.3 Validación de utilidad

> ### 🔧 Métricas de utilidad
> - **Para entrenar modelos:** entrenar en sintéticos y evaluar en reales. $\text{Ratio} = \text{Acc}_{sint} / \text{Acc}_{real} \approx 1$ (ideal).
> - **Para análisis:** replicar el análisis en ambos datasets y comparar coeficientes, magnitudes y significancia.
> - **Para *data augmentation*:** comparar el modelo *baseline* (sólo reales) contra el aumentado (reales + sintéticos) en un test real.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def evaluar_utilidad(X_real, y_real, X_sint, y_sint, test_size=0.3, random_state=42):
    '''Compara accuracy de modelos entrenados en reales, en sintéticos y en la combinación.'''
    X_train, X_test, y_train, y_test = train_test_split(
        X_real, y_real, test_size=test_size, random_state=random_state)

    m_real = LogisticRegression(max_iter=1000, random_state=random_state).fit(X_train, y_train)
    acc_real = accuracy_score(y_test, m_real.predict(X_test))

    m_sint = LogisticRegression(max_iter=1000, random_state=random_state).fit(X_sint, y_sint)
    acc_sint = accuracy_score(y_test, m_sint.predict(X_test))

    X_aug = np.vstack([X_train, X_sint]); y_aug = np.concatenate([y_train, y_sint])
    m_aug = LogisticRegression(max_iter=1000, random_state=random_state).fit(X_aug, y_aug)
    acc_aug = accuracy_score(y_test, m_aug.predict(X_test))

    print(f"Accuracy entrenando en REALES:      {acc_real:.4f}")
    print(f"Accuracy entrenando en SINTÉTICOS:  {acc_sint:.4f}   (ratio = {acc_sint/acc_real:.3f})")
    print(f"Accuracy AUMENTADO (reales+sint):   {acc_aug:.4f}   (mejora = {(acc_aug-acc_real)*100:+.2f} pts)")
    return {"acc_real": acc_real, "acc_sint": acc_sint, "acc_aug": acc_aug}

# Dataset real de clasificación; sintéticos por clase con la normal multivariada
X_c, y_c = make_classification(n_samples=800, n_features=5, n_informative=3, n_redundant=0,
                               n_clusters_per_class=1, class_sep=0.9, flip_y=0.12, random_state=7)

X_sint_list, y_sint_list = [], []
for clase in np.unique(y_c):
    X_clase = X_c[y_c == clase]
    X_clase_sint, _, _ = generar_sinteticos_multivariados(X_clase, len(X_clase), random_state=clase)
    X_sint_list.append(X_clase_sint)
    y_sint_list.append(np.full(len(X_clase), clase))
X_c_sint = np.vstack(X_sint_list)
y_c_sint = np.concatenate(y_sint_list)

_ = evaluar_utilidad(X_c, y_c, X_c_sint, y_c_sint)

---
# 6. Resumen y Mejores Prácticas

1. Los **datos sintéticos** tienen múltiples propósitos: aumentar datos, proteger privacidad, evaluar modelos y balancear clases.
2. **Método paramétrico:** simple y efectivo cuando los datos siguen distribuciones conocidas. Estimar parámetros y muestrear.
3. **Método de perturbación:** añadir ruido controlado a datos reales. Preserva estructura compleja, pero ofrece menor privacidad.
4. **SMOTE:** específico para balanceo; genera sintéticos interpolando entre vecinos cercanos de la clase minoritaria.
5. **GANs:** estado del arte, complejas pero poderosas para datos de alta dimensionalidad y relaciones no lineales.
6. La **validación** es crítica: medir tanto fidelidad (¿parecidos estadísticamente?) como utilidad (¿sirven para el propósito?).
7. Usar **múltiples métricas:** tests KS, Wasserstein, comparación de momentos y matrices de correlación.
8. Para ML: evaluar el **ratio de utilidad** entrenando en sintéticos y evaluando en reales.
9. **Balance:** los sintéticos deben ser realistas pero no copias exactas (privacidad vs. utilidad).

> ### 💡 Cierre
> La generación de datos sintéticos es tanto arte como ciencia. El método apropiado depende del propósito, la complejidad de los datos y los recursos disponibles. Métodos simples como el paramétrico o la perturbación funcionan sorprendentemente bien para muchos casos prácticos. La validación rigurosa es esencial: los datos sintéticos sólo son valiosos si preservan las propiedades relevantes de los reales. En la era del machine learning y las crecientes preocupaciones de privacidad, la capacidad de generar datos sintéticos de alta calidad se consolida como una habilidad cada vez más importante en ciencia de datos.